# Rule Test — 폴더 전체 사례 비교

지정한 폴더 아래의 모든 사례 JSON을 찾아 R00~R11을 판정하고, 합성데이터의 `rule_results`와 일괄 비교합니다.

In [ ]:
import json
import sys
from pathlib import Path

SOURCE_DIR = Path("2026-08-14/src").resolve()
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from vitamin.io import load_case, write_json
from vitamin.pipeline import VitaminPipeline

## 1. 비교할 폴더 선택

`DATA_ROOT`를 비교하려는 사례 폴더로 지정하세요. 모든 하위 폴더를 자동으로 탐색합니다.

In [ ]:
DATA_ROOT = Path("합성데이터")
OUTPUT_DIR = Path("실행결과/rule_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"사례 폴더를 찾을 수 없습니다: {DATA_ROOT.resolve()}")

case_files = []
for path in sorted(DATA_ROOT.rglob("*.json")):
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(payload, dict) and "documents" in payload:
            case_files.append(path)
    except (json.JSONDecodeError, UnicodeDecodeError):
        pass

print("입력 폴더:", DATA_ROOT.resolve())
print("발견한 사례 JSON:", len(case_files))

## 2. 전체 사례 판정 및 정답 비교

In [ ]:
summaries = []

for case_path in case_files:
    source_data = json.loads(case_path.read_text(encoding="utf-8"))
    case = load_case(case_path)
    result = VitaminPipeline(require_review=True).run(case).to_dict()
    actual = {item["rule_id"]: item["status"] for item in result["rules"]}

    expected = None
    matches = None
    if "rule_results" in source_data:
        expected = {
            rule_id: value[f"{rule_id.lower()}_result"]
            for rule_id, value in source_data["rule_results"].items()
        }
        matches = {
            rule_id: actual.get(rule_id) == expected_status
            for rule_id, expected_status in expected.items()
        }

    report_data = dict(result)
    if expected is not None:
        report_data["validation"] = {"expected": expected, "matches": matches}
    write_json(OUTPUT_DIR / f"{case.case_id}_report.json", report_data)

    mismatched_rules = (
        [rule_id for rule_id, matched in matches.items() if not matched]
        if matches is not None else []
    )
    summaries.append({
        "case_id": case.case_id,
        "source": str(case_path),
        "actual": actual,
        "expected": expected,
        "matches": matches,
        "all_rules_match": all(matches.values()) if matches is not None else None,
        "mismatched_rules": mismatched_rules,
        "review_required": result["review_required"],
    })

comparable = [item for item in summaries if item["all_rules_match"] is not None]
matched = [item for item in comparable if item["all_rules_match"]]
summary_data = {
    "total_cases": len(summaries),
    "comparable_cases": len(comparable),
    "all_rules_match_cases": len(matched),
    "case_accuracy": len(matched) / len(comparable) if comparable else None,
    "cases": summaries,
}
summary_path = OUTPUT_DIR / "summary.json"
write_json(summary_path, summary_data)

print("전체 사례:", len(summaries))
print("정답 비교 가능 사례:", len(comparable))
print("R00~R11 전체 일치 사례:", len(matched))
if comparable:
    print("사례 단위 일치율:", f'{summary_data["case_accuracy"]:.1%}')
print("요약 파일:", summary_path.resolve())

## 3. 불일치 사례만 확인

In [ ]:
mismatch_cases = [item for item in comparable if not item["all_rules_match"]]

if not mismatch_cases:
    print("모든 비교 가능 사례가 정답과 일치합니다.")
else:
    print("불일치 사례 수:", len(mismatch_cases))
    for item in mismatch_cases:
        details = []
        for rule_id in item["mismatched_rules"]:
            details.append(
                f'{rule_id}(예상={item["expected"].get(rule_id)}, 실제={item["actual"].get(rule_id)})'
            )
        print(f'{item["case_id"]}: {", ".join(details)}')